# Question 21 - 151. Reverse Words in a String

Given an input string `s`, reverse the order of the **words**.

A **word** is defined as a sequence of non-space characters. The **words** in `s` will be separated by at least one space.

Return *a string of the words in reverse order concatenated by a single space.*

**Note** that `s` may contain leading or trailing spaces or multiple spaces between two words. The returned string should only have a single space separating the words. Do not include any extra spaces.

**Example 1:**

    Input: s = "the sky is blue"
    Output: "blue is sky the"

**Example 2:**

    Input: s = "  hello world  "
    Output: "world hello"
    Explanation: Your reversed string should not contain leading or trailing spaces.

**Example 3:**

    Input: s = "a good   example"
    Output: "example good a"
    Explanation: You need to reduce multiple spaces between two words to a single space in the reversed string.

**Constraints:**

- `1 <= s.length <= 10^4`
- `s` contains English letters (upper-case and lower-case), digits, and spaces `' '`.
- There is **at least one** word in `s`.

**Follow-up:** If the string data type is mutable in your language, can you solve it **in-place** with `O(1)` extra space?

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Split → reverse → join; follow-up: reverse everything, then reverse each word

📘 **Revise first:** [Part 5.4–5.5 · split, join, building strings](./0-concepts-array-and-string.ipynb) · [Pattern F · reverse tricks](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (prepend each word to a string) | O(n²) | O(n) |
| 2️⃣ Optimized (split, reverse, join) | O(n) | O(n) |
| ➕ Follow-up (in place on a character array) | O(n) | O(1) extra |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Reverse the **order of the words** (not the letters inside each word), and tidy up the spacing: exactly one space between words, none at the start or end.

```
"the sky is blue"     → "blue is sky the"
"  hello world  "     → "world hello"
"a good   example"    → "example good a"
```

**Everyday picture: a stack of index cards.** Write each word on a card, laying the cards on the table in reading order. Then pick them up **from the last card to the first** and read them out with one space between. Blank bits of the sentence never become cards, so extra spaces disappear automatically.

In Python:
- "Make the cards" = `s.split()`, which splits on any run of spaces and ignores empty bits.
- "Pick them up in reverse" = `reversed(...)`.
- "Read them out with single spaces" = `" ".join(...)`.

### Step 0 · Clarify before coding

🗣️ *"I reverse the order of the words, not the letters in them. The output has single spaces between words and no leading or trailing spaces."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Extra spaces (leading, trailing, multiple)? | **Yes.** | They must be cleaned up. |
| Keep the letters' case? | **Yes**, unchanged. | |
| Can I modify the string in place? | Python strings **can't be changed** (immutable). | A true in-place version needs a list of characters. That's the follow-up. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** walk through the sentence, pull out each word, and **stick it on the front** of the result: `result = word + " " + result`.

🗣️ *"A naive version extracts words one by one and prepends each to the result string. It reads naturally, but strings are immutable, so every prepend copies the whole result built so far. That's 1 + 2 + 3 + … characters copied, O(n²) in the worst case."*

**The fix to say out loud:** collect words in a **list** (adding to a list is cheap) and **join once** at the end (concepts notebook, Part 5.5).

In [1]:
class SolutionBrute:
    def reverseWords(self, s: str) -> str:
        result = ""
        i, n = 0, len(s)
        while i < n:
            while i < n and s[i] == " ":      # skip spaces
                i += 1
            if i == n:
                break
            j = i
            while j < n and s[j] != " ":      # find the end of this word
                j += 1
            word = s[i:j]
            result = word if not result else word + " " + result   # copies all of `result`
            i = j
        return result

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** the slow part is repeatedly **copying a growing string**. Put the words in a **list**, reverse the list, and **join once**. `join` measures the total size first and copies each character exactly once.

And `s.split()` (no argument) already handles all the messy spacing for us.

🗣️ *"I'll split on whitespace, which also removes the extra spaces, reverse the list of words, and join them with single spaces. Each step is linear, so it's O(n) time and O(n) space for the word list and the output."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Prepend each word to a string | ❌ | Every prepend copies the whole result. O(n²). |
| **`split()` → `reversed` → `" ".join`** | ✅ **best in Python** | Linear, readable, and the spacing rules come for free. |
| Push words onto a `deque` with `appendleft`, then join | ✅ | Also O(n). Handy if you're parsing words manually. |
| A stack of words | ✅ | Same as reversing a list. |
| Regex split | ⚠️ | Works, but adds overhead and edge cases. `split()` is simpler. |
| **In place on a character array (follow-up)** | ✅ for the follow-up | Reverse the whole thing, then reverse each word, then tidy spaces. O(1) extra, but only where strings are mutable. |

**Expect "can you do it without split/join?"** Then show the in-place version below.

In [2]:
class Solution:
    def reverseWords(self, s: str) -> str:
        return " ".join(reversed(s.split()))


class SolutionInPlace:
    # Follow-up: O(1) extra space on a mutable character array (simulated with a list)
    def reverseWords(self, s: str) -> str:
        a = list(s)

        def reverse(lo: int, hi: int) -> None:
            while lo < hi:
                a[lo], a[hi] = a[hi], a[lo]
                lo += 1
                hi -= 1

        # step 1: squeeze out extra spaces (write pointer w, read pointer r)
        w, r, n = 0, 0, len(a)
        while r < n:
            if a[r] != " ":
                if w > 0:
                    a[w] = " "            # exactly one space before each new word
                    w += 1
                while r < n and a[r] != " ":
                    a[w] = a[r]
                    w += 1
                    r += 1
            else:
                r += 1
        del a[w:]                         # trim (in C++/Java you'd just track the length)

        # step 2: reverse everything:  "the sky" -> "yks eht"
        reverse(0, len(a) - 1)

        # step 3: reverse each word back: "yks eht" -> "sky the"
        start = 0
        for i in range(len(a) + 1):
            if i == len(a) or a[i] == " ":
                reverse(start, i - 1)
                start = i + 1
        return "".join(a)

### Step 3 · Toy example walkthrough (tell it like a story)

**Main solution:** `s = "  hello world  "`

| step | result |
|---|---|
| `s.split()` | `["hello", "world"]` (the extra spaces vanish) |
| `reversed(...)` | `["world", "hello"]` |
| `" ".join(...)` | `"world hello"` ✅ |

**Follow-up, in place:** `s = "a good   example"`

| step | characters |
|---|---|
| 1) squeeze spaces | `a good example` |
| 2) reverse everything | `elpmaxe doog a` |
| 3) reverse each word | `example good a` ✅ |

**Narrated:** "Reversing the whole sentence puts the words in the right order, but each word is spelled backwards. So I flip each word on its own to fix the spelling, and the order stays reversed."

In [3]:
cases = [
    ("the sky is blue", "blue is sky the"),
    ("  hello world  ", "world hello"),
    ("a good   example", "example good a"),
    ("single", "single"),
    ("   x   ", "x"),
    ("1 2 3", "3 2 1"),
]
for s, expected in cases:
    for Impl in (SolutionBrute, Solution, SolutionInPlace):
        assert Impl().reverseWords(s) == expected, Impl.__name__
    print(repr(s), "->", repr(expected))
print("All tests passed ✅")

'the sky is blue' -> 'blue is sky the'
'  hello world  ' -> 'world hello'
'a good   example' -> 'example good a'
'single' -> 'single'
'   x   ' -> 'x'
'1 2 3' -> '3 2 1'
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Prepend to a string | O(n²) | O(n) | Each word copies everything built so far. |
| **split / reverse / join** | **O(n)** | **O(n)** | Each character is copied a constant number of times. |
| In place on a char array | O(n) | O(1) | Three passes with swaps; no new words list. |

**Why O(n) time (main solution)?** `split` reads each character once, `reversed` touches each word once, and `join` writes each character once.

**Why O(n) memory?** The list of words and the output string together hold about n characters.

**Edge cases to mention:** a single word, spaces only around one word, several spaces between words.

---

#### 🗣️ Full interview script (about 45 seconds)

*"I need the words in reverse order with single spaces and no extra spaces at the ends.*

*Building the result by prepending words would copy the string every time, which is O(n²). Instead, I split on whitespace, which also normalises the spacing, reverse the list, and join with single spaces. That's O(n) time and O(n) space.*

*If asked to do it in place on a mutable character array: squeeze out the extra spaces, reverse the whole array, then reverse each word back. O(n) time, O(1) extra space."*